In [23]:
!git clone https://github.com/gauriiiiiiiiiiii/PatientTrialMapper.git

Cloning into 'PatientTrialMapper'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 43 (delta 17), reused 35 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 32.87 KiB | 8.22 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [24]:
import os
os.chdir('/kaggle/working/PatientTrialMapper')
print("Current directory:", os.getcwd())

Current directory: /kaggle/working/PatientTrialMapper


In [26]:
!pip install -r requirements.txt

In [27]:
!nvidia-smi

Sun Jun  7 06:14:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [28]:
!git clone https://github.com/gauriiiiiiiiiiii/PatientTrialMapper.git /kaggle/working/PatientTrialMapper
%cd /kaggle/working/PatientTrialMapper
!ls

fatal: destination path '/kaggle/working/PatientTrialMapper' already exists and is not an empty directory.
/kaggle/working/PatientTrialMapper
'=0.46.1'    data_prep.py   pages		 __pycache__
 app.py      Dockerfile     PatientTrialMapper	 README.md
 config.py   evaluate.py    predict.py		 requirements.txt
 data	     models	    push_to_hub.py	 train.py


In [29]:
!python data_prep.py

Built-in examples: 20

Generating synthetic data (skipped if no API key)...
OPENAI_API_KEY not set in config.py — skipping synthetic generation.

── Label distribution ──
response
Eligible        10
Not Eligible    10

── Prompt length stats ──
  min=303  max=371  mean=330

EDA plot saved → data/eda.png

Split → train:16  val:2  test:2
Saving the dataset (1/1 shards): 100%|████| 2/2 [00:00<00:00, 864.18 examples/s]

Saved:
  data/train
  data/val
  data/test_samples.json

data_prep.py done. Run: python train.py


In [30]:
!pip install trl -q

In [31]:
!pip install -U bitsandbytes>=0.46.1 -q

In [37]:
!git pull
!python train.py

Already up to date.
GPU: Tesla T4  (15.6 GB VRAM)

Loading datasets...
  Train: 16  Val: 2

Loading tokenizer...

Loading base model: mistralai/Mistral-7B-Instruct-v0.3
(This downloads ~14 GB on first run — be patient)
Loading weights: 100%|█| 291/291 [00:08<00:00, 33.63it/s, Materializing param=mo

Applying QLoRA...
trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754

Setting up trainer...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.

Training started...
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
{'loss': '0.5487', 'grad_norm': '0.7656', 'learning_rate': '0.0001998', 'entropy': '0.5622', 'num_tokens': '3.554e+04', 'mean_token_accuracy': '0.

In [38]:
import os
# Check if adapters were saved
if os.path.exists('/kaggle/working/PatientTrialMapper/models/lora_adapters'):
    files = os.listdir('/kaggle/working/PatientTrialMapper/models/lora_adapters')
    print("Adapters saved:", files)
else:
    print("Adapters NOT found - training may have crashed")
    
# Check checkpoints
if os.path.exists('/kaggle/working/PatientTrialMapper/models/checkpoints'):
    ckpts = os.listdir('/kaggle/working/PatientTrialMapper/models/checkpoints')
    print("Checkpoints:", ckpts)

Adapters saved: ['README.md', 'tokenizer.json', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json']
Checkpoints: ['README.md', 'checkpoint-500', 'checkpoint-300', 'checkpoint-200', 'checkpoint-400', 'checkpoint-100']


In [39]:
!python evaluate.py

Test set loaded: 2 examples

Loading fine-tuned model...
Loading weights: 100%|█| 291/291 [00:04<00:00, 62.98it/s, Materializing param=mo

Running predictions on 2 test examples...
  [1/2]  True: Not Eligible     Pred: Eligible
  [2/2]  True: Eligible         Pred: Eligible

  EVALUATION RESULTS
  Total test samples : 2

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/skl

In [41]:
import shutil
shutil.make_archive('/kaggle/working/lora_adapters', 'zip', 
                    '/kaggle/working/PatientTrialMapper/models/lora_adapters')
print("Download this: /kaggle/working/lora_adapters.zip")

Download this: /kaggle/working/lora_adapters.zip
